# Lab21 — Khôi phục checkpoint và chạy bổ sung
Nguyễn Nhân Sâm · 2A202602672. Chọn T4 và chạy từng ô từ trên xuống.

Upload đúng `lab21_2A202602672_core_results_20261007T124407Z.zip` (~478 MB).
Core 1–7 và các ZIP gốc đã được giữ. Notebook này đo bổ sung 15 câu regression,
merge/hot-swap (B1), quét rank8/16/64 (B4); không thay baseline/verdict core.

Mỗi lần chạy ô setup tạo thư mục attempt mới. Cửa sổ mặc định là **2 giờ kể từ khi
khai báo hàm chạy**, không dùng mốc23:20 ngày07/10 nữa. Có thể tăng RUN_WINDOW_HOURS
trước khi chạy ô khai báo nếu cần thêm thời gian. Guard kiểm tra trước từng bước,
không cam kết ngắt lệnh CUDA đang chạy tại một thời điểm cố định.

**ZIP được tạo không có nghĩa đã hoàn tất.** Ô chỉ in HOÀN TẤT sau khi exit0, status
complete, seal_unchanged=true và đủ file/50–15 câu. Nếu lỗi/incomplete, ZIP log vẫn
được tải rồi ô báo lỗi đỏ để bạn biết cần kiểm tra. Giữ và gửi tất cả ZIP được tải.

Model/tokenizer revision gốc không được ghi nên lần đo mới là supplementary repeat.
Ca thua regression phải được ghi đúng nhóm, không gọi thành ca thua target.
B2/B3/B5 và phản tư cá nhân cần thực hiện riêng; notebook không chứng minh đủ115điểm.


In [ ]:
import pathlib, subprocess, sys, os, json, hashlib, datetime, zipfile, io, urllib.request, uuid
ATTEMPT=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'-'+uuid.uuid4().hex[:8]
ROOT=pathlib.Path('/content/Lab21-followup-'+ATTEMPT)
SOURCE='90679f91842e9f42c9192b9c5eab316935705e9c'
if not ROOT.exists():
    subprocess.run(['git','clone','--no-checkout','https://github.com/Nguyen-Sam-sheep-zzz/Day21-Track3-Finetuning-Lab.git',str(ROOT)],check=True)
    subprocess.run(['git','checkout','--detach',SOURCE],cwd=ROOT,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()==SOURCE
os.chdir(ROOT)
os.environ.update(COMPUTE_TIER='T4',EPOCHS='2',MASK_MODE='assistant-only')
os.environ.pop('BASE_MODEL',None); os.environ.pop('EVAL_LIMIT',None)
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Cần runtime T4.'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)

In [ ]:
from google.colab import files
EXPECTED_NAME='lab21_2A202602672_core_results_20261007T124407Z.zip'
EXPECTED_SHA='78f715550078cf33e3906e3df50ce23164ee259182fb29b2b83887519a407b1a'
uploaded=files.upload()
assert set(uploaded)=={EXPECTED_NAME}, 'Chỉ chọn đúng ZIP core_results gốc.'
raw=uploaded[EXPECTED_NAME]
assert hashlib.sha256(raw).hexdigest()==EXPECTED_SHA, 'ZIP khác checkpoint đã kiểm tra.'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(set(z.namelist()))==len(z.namelist())
    for info in z.infolist():
        p=pathlib.PurePosixPath(info.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in info.filename
        assert info.filename.startswith('lab21_2A202602672/')
        assert ((info.external_attr >> 16) & 0o170000) != 0o120000, 'Không nhận symlink.'
        if info.is_dir(): continue
        rel=info.filename.removeprefix('lab21_2A202602672/')
        if not rel.startswith(('results/','adapters/correct/','adapters/attn_only/')): continue
        dest=ROOT/rel; content=z.read(info)
        if dest.exists() and dest.read_bytes()!=content:
            raise RuntimeError('Runtime có artifact khác: '+rel+'. Dùng runtime mới để giữ bằng chứng.')
        dest.parent.mkdir(parents=True,exist_ok=True); dest.write_bytes(content)
del uploaded,raw
print('Đã kiểm tra SHA256 và khôi phục 2 adapter + results.')

In [ ]:
manifest=json.loads((ROOT/'results/experiment_manifest.json').read_text(encoding='utf-8'))
versions=manifest['environment']
packages=['transformers','tokenizers','peft','trl','accelerate','datasets','bitsandbytes','safetensors']
pins=[name+'=='+versions[name] for name in packages]
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt',*pins],check=True)
sys.path.insert(0,str(ROOT/'src'))
from labkit import data,evidence,config
seed_rows=[json.loads(s) for s in (ROOT/'data/train_seed.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
train,val=data.split(seed_rows,train_frac=.9,seed=42)
(ROOT/'data/split').mkdir(exist_ok=True)
for name,rows in [('train',train),('val',val)]:
    (ROOT/f'data/split/{name}.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows),encoding='utf-8')
for name,expected in manifest['context']['file_sha256'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==expected, 'Dataset/split khác checkpoint: '+name
print('Nguồn, split seed42 và checksum eval đúng; version mới sẽ được ghi vào bằng chứng bổ sung.')

In [ ]:
HELPER_COMMIT='384a8aeee7831ff8c68bf0fec1cb55205df08a71'
HELPER_SHA='594267585022da4be5d002af50301cad49043ab388e197a99a018c8dbdcc8bef'
url='https://raw.githubusercontent.com/Nguyen-Sam-sheep-zzz/Day21-Track3-Finetuning-Lab/'+HELPER_COMMIT+'/student_tools/deadline_experiments.py'
helper=urllib.request.urlopen(url).read()
assert hashlib.sha256(helper).hexdigest()==HELPER_SHA, 'Helper khác bản đã kiểm tra.'
(ROOT/'student_tools').mkdir(exist_ok=True)
(ROOT/'student_tools/deadline_experiments.py').write_bytes(helper)
(ROOT/'followup_helper_receipt.json').write_text(json.dumps({'helper_commit':HELPER_COMMIT,'helper_sha256':HELPER_SHA,'core_source_commit':SOURCE},indent=2),encoding='utf-8')
print('Helper cố định:',HELPER_COMMIT)

In [ ]:
RUN_WINDOW_HOURS=2
STOP_AT=(datetime.datetime.now(datetime.timezone.utc)+datetime.timedelta(hours=RUN_WINDOW_HOURS)).isoformat()
print('Cửa sổ chạy đến UTC:',STOP_AT,'; hours:',RUN_WINDOW_HOURS)

def download_evidence(stage):
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest=ROOT/f'lab21_2A202602672_followup_{stage}_{stamp}.zip'
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
        for folder in ['bonus_results','bonus_logs']:
            for p in sorted((ROOT/folder).rglob('*')):
                if p.is_file(): z.write(p,'lab21_2A202602672/'+p.relative_to(ROOT).as_posix())
        z.write(ROOT/'followup_helper_receipt.json','lab21_2A202602672/followup_helper_receipt.json')
    print('Evidence ZIP:',dest.name,'SHA256:',hashlib.sha256(dest.read_bytes()).hexdigest())
    files.download(str(dest))

def require_measurement(name):
    path=ROOT/'bonus_results'/name
    if not path.is_file(): raise RuntimeError('Chưa có file kết quả: '+name)
    return json.loads(path.read_text(encoding='utf-8'))

def require_full_target(result):
    if result.get('n')!=50 or len(result.get('rows',[]))!=50:
        raise RuntimeError('Kết quả chưa đánh giá đủ 50 câu target.')

def verify_completion(stage,rc):
    if rc!=0: raise RuntimeError(f'{stage} chưa hoàn tất (exit {rc}). ZIP log đã được tải; gửi ZIP để kiểm tra.')
    record=require_measurement(f'run_status_{stage}.json')
    if record.get('stage')!=stage or record.get('result',{}).get('status')!='complete':
        raise RuntimeError('Status chưa complete. ZIP được tạo chỉ lưu trạng thái/log, không chứng minh chạy xong.')
    if record.get('seal_unchanged') is not True:
        raise RuntimeError('Kiểm tra giữ nguyên core chưa đạt; dừng để kiểm tra ZIP.')
    if stage=='regression':
        result=require_measurement('regression_followup.json')
        if result.get('status')!='complete' or result.get('n')!=15 or len(result.get('rows',[]))!=15:
            raise RuntimeError('Chưa có đủ 15 output regression.')
    elif stage=='b1':
        merge=require_measurement('merge_check.json')
        require_full_target(merge.get('before',{}));require_full_target(merge.get('after',{}))
        if merge.get('status')!='complete' or merge.get('passed') is not True:
            raise RuntimeError('Đã có phép đo nhưng merge chưa đạt cổng không suy giảm; không xác nhận B1.')
        swap=require_measurement('hot_swap.json')
        identity=swap.get('same_base_identity')
        if swap.get('status')!='complete' or not identity:
            raise RuntimeError('Chưa có hot-swap hoàn chỉnh trên cùng base.')
        for key in ('correct','attn_only'):
            result=swap.get('adapters',{}).get(key,{})
            require_full_target(result)
            if result.get('base_identity')!=identity:
                raise RuntimeError('Hot-swap không xác nhận cùng base.')
    elif stage=='b4':
        sweep=require_measurement('rank_sweep.json')
        if sweep.get('status')!='complete' or set(sweep.get('ranks',{}))!={'8','16','64'}:
            raise RuntimeError('Rank sweep còn thiếu rank hoặc chưa complete.')
        for result in sweep['ranks'].values():require_full_target(result)
    else: raise RuntimeError('Stage không hợp lệ.')

def run_stage(stage):
    if stage not in ('regression','b1','b4'):raise ValueError('Stage không hợp lệ.')
    (ROOT/'bonus_logs').mkdir(exist_ok=True)
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log=ROOT/f'bonus_logs/{stage}_{stamp}.txt'
    cmd=[sys.executable,'-u','student_tools/deadline_experiments.py','--stages',stage,'--stop-at-utc',STOP_AT]
    try:
        with log.open('w',encoding='utf-8') as f:
            p=subprocess.Popen(cmd,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in p.stdout:
                print(line,end='');f.write(line);f.flush()
            rc=p.wait()
        print('Stage',stage,'exit',rc)
    finally:
        download_evidence(stage)
    verify_completion(stage,rc)
    print('HOÀN TẤT:',stage,'— đã có đủ file kết quả và core không đổi. Giữ ZIP để kiểm tra/report.')


## 1. Đo regression trước — ưu tiên bằng chứng cho mục 3.4
Không huấn luyện lại. Phép đo 15 câu và so với raw baseline đã đóng băng.

In [ ]:
run_stage('regression')

## 2. B1 — merge và đổi 2 adapter trên cùng base
Đo đủ 50 câu trước/sau merge; đo correct và attn_only trên cùng một model base.

In [ ]:
run_stage('b1')

## 3. B4 — Quét rank8/16/64
r16 dùng adapter cũ nhưng đánh giá lại; r8/r64 huấn luyện cùng recipe.
Mỗi rank mới cần15phút dự phòng trước khi bắt đầu. Chỉ nhận hoàn tất khi cả ba rank
có đủ50output. Bước này có thể mất17–25phút hoặc lâu hơn; giữ Colab kết nối đến khi
tải được ZIP. Nếu guard thời gian từ chối, ô sẽ báo lỗi/incomplete, không báo thành công.


In [ ]:
run_stage('b4')

## Kết thúc
Giữ từng ZIP đã tải và gửi để kiểm tra/cập nhật report. Không bấm lại stage trong cùng
attempt vì helper bảo vệ file cũ. Nếu cần lặp lại, mở một runtime mới hoặc chạy lại ô
setup để tạo attempt riêng, sau đó upload checkpoint và chạy lại các ô chuẩn bị.
Đừng chạy lại ô setup khi một stage vẫn đang hoạt động.

Không cần train lại core. Phản tư cá nhân cần Sâm đọc/chỉnh. B2/B3/B5 thực hiện riêng.
Các ZIP và báo cáo gốc được bảo toàn; thiếu bằng chứng hoặc cổng merge FAIL vẫn phải công bố.
